# HIPAA Breach Data - Feature Engineering
Author: Wendy Lacan

**Input:** data/processed/breach_report_cleaned.csv

**Output:** data/processed/breach_report_features.csv

### 1. Load Cleaned Data

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/breach_report_cleaned.csv")
df['breach_submission_date'] = pd.to_datetime(df['breach_submission_date'])

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Loaded: 6035 rows, 11 columns


,name_of_covered_entity,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,year,month
0,Ko-Kwel Wellness Center,OR,Healthcare Provider,543,2026-02-03,Hacking/IT Incident,Network Server,Yes,NaN,2026,2
1,Chattanooga C.A.R.E.S. d/b/a Cempa Community Care,TN,Healthcare Provider,1341,2026-01-30,Unauthorized Access/Disclosure,Other,Yes,NaN,2026,1
2,Baltimore City Health Department,MD,Healthcare Provider,2597,2026-01-28,Hacking/IT Incident,Network Server,Yes,NaN,2026,1
3,Deschutes County Health Services,OR,Healthcare Provider,1305,2026-01-22,Hacking/IT Incident,Network Server,Yes,NaN,2026,1
4,Bay Area Community Health,CA,Healthcare Provider,9912,2026-01-16,Hacking/IT Incident,Network Server,Yes,"The covered entity (CE), Bay Area Community He...",2026,1


### 2. Breach Type Encoding

#### 2.1 Multi-Label Binarization

In [13]:
breach_type_dummies = (
    df['type_of_breach']
    .str.split(', ')
    .explode()
    .str.strip()
    .pipe(lambda s: pd.get_dummies(s))
    .groupby(level=0)
    .max()
)

breach_type_dummies.columns = [
    "breach_type_" + col.lower().replace("/", "_").replace(" ", "_")
    for col in breach_type_dummies.columns
]

df = df.join(breach_type_dummies)
print("Breach type columns added:", list(breach_type_dummies.columns))

Breach type columns added: ['breach_type_hacking_it_incident', 'breach_type_improper_disposal', 'breach_type_loss', 'breach_type_other', 'breach_type_theft', 'breach_type_unauthorized_access_disclosure', 'breach_type_unknown']


Each branch type is now represented as a binary indicator column. A record can have multiple breach types active at the same time, reflecting the original multi-value structure of the source data. 

#### 2.2 Primary Breach Type Flag

In [14]:
def primary_breach_type(val):
    if pd.isna(val):
        return 'Unknown'
    return val.split(',')[0].strip()

df['primary_breach_type'] = df['type_of_breach'].apply(primary_breach_type)
df['primary_breach_type'].value_counts()

primary_breach_type
Hacking/IT Incident               3649
Unauthorized Access/Disclosure    1431
Theft                              631
Loss                               179
Improper Disposal                   98
Other                               44
Unknown                              3
Name: count, dtype: int64

### 3. Location Encoding

#### 3.1 Location Multi-Label Binarization

In [15]:
location_dummies = (
    df['location_of_breached_information']
    .str.split(', ')
    .explode()
    .str.strip()
    .pipe(lambda s: pd.get_dummies(s))
    .groupby(level=0)
    .max()
)

location_dummies.columns = [
    "location_" + col.lower().replace("/", "_").replace(" ", "_")
    for col in location_dummies.columns
]

df = df.join(location_dummies)

df = df.rename(columns={'location_of_breached_information': 'raw_location_of_breached_information'})
print("Location columns added:", list(location_dummies.columns))

Location columns added: ['location_desktop_computer', 'location_electronic_medical_record', 'location_email', 'location_laptop', 'location_network_server', 'location_other', 'location_other_portable_electronic_device', 'location_paper_films']


#### 3.2 Primary Location Flag

In [16]:
def primary_location(val):
    if pd.isna(val):
        return 'Unknown'
    return val.split(',')[0].strip()

df['primary_location'] = df['raw_location_of_breached_information'].apply(primary_location)
df['primary_location'].value_counts()

primary_location
Network Server                      2533
Email                               1501
Paper/Films                          724
Electronic Medical Record            347
Laptop                               276
Desktop Computer                     264
Other                                222
Other Portable Electronic Device     168
Name: count, dtype: int64

### 4. Breach Size Features

#### 4.1 Mega-Breach Flag

In [17]:
df['is_mega_breach'] = (df['individuals_affected'] > 1_000_000).astype(int)
print(f"Mega-breaches: {df['is_mega_breach'].sum()} ({df['is_mega_breach'].mean():.1%} of records)")

Mega-breaches: 102 (1.7% of records)


#### 4.2 Breach Size Tier

In [18]:
def breach_size_tier(n):
    if n < 10_000:
        return 'Small'
    elif n < 100_000:
        return 'Medium'
    elif n < 1_000_000:
        return 'Large'
    else:
        return 'Mega'
    
df['breach_size_tier'] = df['individuals_affected'].apply(breach_size_tier)

tier_order = ['Small', 'Medium', 'Large', 'Mega']
df['breach_size_tier'] = pd.Categorical(
    df['breach_size_tier'],
    categories=tier_order,
    ordered=True
)

df['breach_size_tier'].value_counts().sort_index()

breach_size_tier
Small     3929
Medium    1503
Large      501
Mega       102
Name: count, dtype: int64

Breach size tiers were defined based on order-of-magnitude thresholds rather than statistical percentiles, since the distribution is heavily right-skewed. This makes tiers more interpretable from a compliance risk standpoint. 

### 4.3 Log-Transformed Individuals Affected

In [19]:
df['log_individuals_affected'] = np.log1p(df['individuals_affected'])

Log transformation compresses the extreme right skew caused by mega-breaches, making **individuals_affected** more suitable for trend analysis and visualization later on. **log1p** is used to safely handle any zero values. 

### 5. Temporal Features

#### 5.1 Quarter Extraction

In [20]:
df['quarter'] = df['breach_submission_date'].dt.quarter

df['quarter'].value_counts().sort_index()

quarter
1    1485
2    1542
3    1487
4    1521
Name: count, dtype: int64

#### 5.2 Days Since Epoch

In [21]:
epoch = pd.Timestamp('2013-01-09')
df['days_since_epoch'] = (df['breach_submission_date'] - epoch).dt.days

print(f"Range: {df['days_since_epoch'].min()} to {df['days_since_epoch'].max()} days")

Range: -7 to 4773 days


-7 means there are records with breach submission dates before the stated dataset start date. 

In [22]:
df[df['days_since_epoch'] < 0][['name_of_covered_entity', 'breach_submission_date', 'state']]

,name_of_covered_entity,breach_submission_date,state
6029,SilverScript Insurance Company,2013-01-08,AZ
6030,WorkflowOne,2013-01-08,OH
6031,University of Nevada School of Medicine,2013-01-08,NV
6032,"Clearpoint Design, Inc.",2013-01-07,MA
6033,"Calvin Schuster,MD",2013-01-04,CA
6034,Group Health Incorporated,2013-01-02,NY


In [23]:
df = df[df['days_since_epoch'] >= 0].reset_index(drop=True)
print(f"Rows after date filter: {len(df)}")

Rows after date filter: 6029


6 records with breach submission dates prior to January 9, 2013 were removed to maintain consistency with the dataset boundary defined notebook 1.

### 6. Entity Features

#### 6.1 Business Associate Flag (binary encode)

In [24]:
df['ba_present'] = (df['business_associate_present'] == 'Yes').astype(int)

print(f"BA involvement rate: {df['ba_present'].mean():.1%}")

BA involvement rate: 30.4%


#### 6.2 Entity Type Encoding

In [25]:
entity_dummies = pd.get_dummies(
    df['covered_entity_type'],
    prefix='entity'
)

entity_dummies.columns = [
    col.lower().replace(" ", "_")
    for col in entity_dummies.columns
]

df = df.join(entity_dummies)
print("Entity type columns added:", list(entity_dummies.columns))

Entity type columns added: ['entity_business_associate', 'entity_health_plan', 'entity_healthcare_clearing_house', 'entity_healthcare_provider']


#### 6.3 State Region Mapping

In [26]:
region_map = {
    #Northeast
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast',
    'NH': 'Northeast', 'RI': 'Northeast', 'VT': 'Northeast',
    'NJ': 'Northeast', 'NY': 'Northeast', 'PA': 'Northeast',

    #Midwest
    'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest',
    'OH': 'Midwest', 'WI': 'Midwest', 'IA': 'Midwest',
    'KS': 'Midwest', 'MN': 'Midwest', 'MO': 'Midwest',
    'NE': 'Midwest', 'ND': 'Midwest', 'SD': 'Midwest',

    # South
    'DE': 'South', 'FL': 'South', 'GA': 'South',
    'MD': 'South', 'NC': 'South', 'SC': 'South',
    'VA': 'South', 'DC': 'South', 'WV': 'South',
    'AL': 'South', 'KY': 'South', 'MS': 'South',
    'TN': 'South', 'AR': 'South', 'LA': 'South',
    'OK': 'South', 'TX': 'South',

    #West
    'AZ': 'West', 'CO': 'West', 'ID': 'West',
    'MT': 'West', 'NV': 'West', 'NM': 'West',
    'UT': 'West', 'WY': 'West', 'AK': 'West',
    'CA': 'West', 'HI': 'West', 'OR': 'West',
    'WA': 'West',

    #Territories
    'PR': 'Territory', 'GU': 'Territory', 'VI': 'Territory'

}

df['region'] = df['state'].map(region_map)

print(df['region'].value_counts())
print(f"\nUnmapped states: {df['region'].isna().sum()}")

region
South        2075
Midwest      1424
West         1295
Northeast    1203
Territory      32
Name: count, dtype: int64

Unmapped states: 0


States are grouped into four U.S. Census regions plus a Territory category for Puerto Rico, Guam, and the US Virgin Islands. Regional grouping reduces the size from 50+ states to 5 categories. 

### 7. HIPAA Control Gap Mapping

#### 7.1 Define Control Gap Categories

In [27]:
breach_type_gap_map = {
    'Hacking/IT Incident': 'Technical Safeguard Failure',
    'Unauthorized Access/Disclosure': 'Administrative Safeguard Failure',
    'Theft': 'Physical/Technical Safeguard Failure',
    'Loss': 'Physical/Technical Safeguard Failure',
    'Improper Disposal': 'Physical/Administrative Safeguard Failure',
    'Other': 'Unclassified',
    'Unknown': 'Unclassified'
}

location_gap_map = {
    # Technical
    'Network Server': 'Technical Safeguard Failure',
    'Electronic Medical Record': 'Technical Safeguard Failure',
    'Electronic Health Record': 'Technical Safeguard Failure',
    'Email': 'Technical/Administrative Safeguard Failure',
    'Other Electronic Media': 'Technical Safeguard Failure',
    'CD/DVD': 'Technical Safeguard Failure',
    'Imaging': 'Technical Safeguard Failure',

    # Physical & Technical
    'Laptop': 'Physical/Technical Safeguard Failure',
    'Desktop Computer': 'Physical/Technical Safeguard Failure',
    'Other Portable Electronic Device': 'Physical/Technical Safeguard Failure',
    'X-Ray Films': 'Physical Safeguard Failure',

    # Physical
    'Paper/Films': 'Physical Safeguard Failure',
    
    # Fallback
    'Other': 'Unclassified'
}

Each breach type and location is mapped to a HIPAA Security Rule safeguard category based on the most likely underlying control failure. The mappings are analytical approximations informed by HHS guidance which reflect probable control gaps rather than official OCR determinations. 

#### 7.2 Map Breach Types to Gap Categories

In [28]:
df['breach_type_gap'] = df['primary_breach_type'].map(breach_type_gap_map)

unmapped = df[df['breach_type_gap'].isna()]['primary_breach_type'].unique()
if len(unmapped) > 0:
    print(f"Unmapped breach types: {unmapped}")
else:
    print(f"All breach types mapped successfully")

df['breach_type_gap'].value_counts()

All breach types mapped successfully


breach_type_gap
Technical Safeguard Failure                  3648
Administrative Safeguard Failure             1429
Physical/Technical Safeguard Failure          808
Physical/Administrative Safeguard Failure      97
Unclassified                                   47
Name: count, dtype: int64

In [29]:
df['location_gap'] = df['primary_location'].map(location_gap_map).fillna('Unclassified')

unmapped = df[df['location_gap'] == 'Unclassified']['primary_location'].unique()
unmapped = [v for v in unmapped if v != 'Other']
if len(unmapped) > 0:
    print(f"Unmapped locations (fell back to 'Unclassified'): {unmapped}")
else:
    print(f"All locations mapped successfully")

df['location_gap'].value_counts()

All locations mapped successfully


location_gap
Technical Safeguard Failure                   2879
Technical/Administrative Safeguard Failure    1501
Physical Safeguard Failure                     720
Physical/Technical Safeguard Failure           707
Unclassified                                   222
Name: count, dtype: int64

### 8. Export

In [30]:
df.to_csv('../data/processed/breach_report_features.csv', index=False)
print(f"Exported: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"New feature columns: {df.shape[1]-9}")

Exported: 6029 rows, 41 columns
New feature columns: 32


In [31]:
feature_groups = {
    'Breach Type': [c for c in df.columns if c.startswith('breach_type_') and not c.endswith('_gap')],
    'Location': [c for c in df.columns if c.startswith('location_') and not c.endswith('_gap')],
    'Entity': [c for c in df.columns if c.startswith('entity_')],
    'Temporal': ['year', 'month', 'quarter', 'days_since_epoch'],
    'Size': ['is_mega_breach', 'breach_size_tier', 'log_individuals_affected'], 
    'Gap Mapping': ['breach_type_gap', 'location_gap']
}

for group, cols in feature_groups.items():
    print(f"\n{group} ({len(cols)} features):")
    for c in cols: print(f" {c}")


Breach Type (7 features):
 breach_type_hacking_it_incident
 breach_type_improper_disposal
 breach_type_loss
 breach_type_other
 breach_type_theft
 breach_type_unauthorized_access_disclosure
 breach_type_unknown

Location (8 features):
 location_desktop_computer
 location_electronic_medical_record
 location_email
 location_laptop
 location_network_server
 location_other
 location_other_portable_electronic_device
 location_paper_films

Entity (4 features):
 entity_business_associate
 entity_health_plan
 entity_healthcare_clearing_house
 entity_healthcare_provider

Temporal (4 features):
 year
 month
 quarter
 days_since_epoch

Size (3 features):
 is_mega_breach
 breach_size_tier
 log_individuals_affected

Gap Mapping (2 features):
 breach_type_gap
 location_gap
